In [4]:
import heapq
import math

# Hücre koordinatları (row, col)
def heuristic(a, b, kind="manhattan"):
    (x1, y1), (x2, y2) = a, b
    if kind == "euclidean":
        return math.hypot(x2 - x1, y2 - y1)
    # default: manhattan
    return abs(x1 - x2) + abs(y1 - y2)

def neighbors(node, grid, allow_diagonal=False):
    (r, c) = node
    rows, cols = len(grid), len(grid[0])
    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if allow_diagonal:
        steps += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    result = []
    for dr, dc in steps:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0:
            result.append((nr, nc))
    return result

def reconstruct_path(came_from, current):
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path

def astar(grid, start, goal, heuristic_kind="manhattan", allow_diagonal=False):
    """
    grid: 2D list where 0 = free cell, 1 = obstacle
    start, goal: (row, col)
    returns: path (list of nodes) or None if no path
    """
    open_set = []
    # heap elements: (f_score, g_score, node)
    g_score = {start: 0}
    f_score = {start: heuristic(start, goal, heuristic_kind)}
    heapq.heappush(open_set, (f_score[start], g_score[start], start))

    came_from = {}

    closed_set = set()

    while open_set:
        _, current_g, current = heapq.heappop(open_set)

        if current == goal:
            return reconstruct_path(came_from, current)

        if current in closed_set:
            continue
        closed_set.add(current)

        for neighbor in neighbors(current, grid, allow_diagonal):
            tentative_g = g_score[current] + math.hypot(neighbor[0]-current[0], neighbor[1]-current[1])
            # if only 4-directional, distance is 1; with diagonal we used hypot (√2)
            if neighbor in g_score and tentative_g >= g_score[neighbor]:
                continue  # not a better path

            # this path is the best until now
            came_from[neighbor] = current
            g_score[neighbor] = tentative_g
            f = tentative_g + heuristic(neighbor, goal, heuristic_kind)
            f_score[neighbor] = f
            heapq.heappush(open_set, (f, tentative_g, neighbor))

    return None  # no path found

def print_grid_with_path(grid, path, start, goal):
    chars = {0: "·", 1: "█"}
    grid_vis = [[chars[cell] for cell in row] for row in grid]
    if path:
        for (r, c) in path:
            if (r, c) == start:
                grid_vis[r][c] = "S"
            elif (r, c) == goal:
                grid_vis[r][c] = "G"
            else:
                grid_vis[r][c] = "*"
    for row in grid_vis:
        print(" ".join(row))

if __name__ == "__main__":
    # Örnek ızgara: 0 = boş, 1 = engel
    example_grid = [
        [0,0,0,0,0,0,0,0],
        [0,1,1,1,0,1,1,1],
        [0,0,0,1,0,1,0,0],
        [0,1,0,0,0,1,0,0],
        [0,1,0,1,0,0,0,0],
        [0,0,0,1,0,1,1,0],
        [0,1,0,0,0,0,0,1],
        [0,0,0,0,1,0,0,0],
    ]
    start = (0, 0)
    goal = (7, 7)

    path = astar(example_grid, start, goal, heuristic_kind="manhattan", allow_diagonal=False)
    print("Bulunan yol (satır, sütun) biçiminde:", path)
    print()
    print_grid_with_path(example_grid, path, start, goal)

Bulunan yol (satır, sütun) biçiminde: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (6, 5), (6, 6), (7, 6), (7, 7)]

S * * * * · · ·
· █ █ █ * █ █ █
· · · █ * █ · ·
· █ · · * █ · ·
· █ · █ * · · ·
· · · █ * █ █ ·
· █ · · * * * █
· · · · █ · * G


In [5]:
import pygame
import math

# Ayarlar
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
CAR_WIDTH = 40
CAR_HEIGHT = 80

class OtonomArac:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.angle = 0  # Aracın yönü (derece)
        self.velocity = 0
        self.steering_angle = 0 # Direksiyon açısı
        
        # Orijinal resim (Sprite)
        self.original_image = pygame.Surface((CAR_WIDTH, CAR_HEIGHT), pygame.SRCALPHA)
        pygame.draw.rect(self.original_image, (0, 0, 255), (0, 0, CAR_WIDTH, CAR_HEIGHT))
        self.image = self.original_image

    def update(self):
        # Basit Kinematik Model (Bicycle Model)
        # Hız ve yöne göre yeni konumu hesapla
        self.x += self.velocity * math.sin(math.radians(self.angle))
        self.y -= self.velocity * math.cos(math.radians(self.angle))
        
        # Direksiyon açısına göre aracın dönmesi
        # Gerçekte: açı += (hız / tekerlek_mesafesi) * tan(direksiyon_açısı)
        self.angle += self.steering_angle * self.velocity * 0.1 

    def draw(self, screen):
        # Aracı döndürerek çiz
        rotated_image = pygame.transform.rotate(self.original_image, -self.angle)
        rect = rotated_image.get_rect(center=(self.x, self.y))
        screen.blit(rotated_image, rect.topleft)
        
        # Sensörleri çiz (Lidar Simülasyonu)
        self.draw_sensors(screen)

    def draw_sensors(self, screen):
        # 5 adet Raycast (Işın) simülasyonu
        sensor_angles = [-30, -15, 0, 15, 30]
        sensor_length = 150
        
        for s_angle in sensor_angles:
            rad_angle = math.radians(self.angle + s_angle)
            end_x = self.x + math.sin(rad_angle) * sensor_length
            end_y = self.y - math.cos(rad_angle) * sensor_length
            pygame.draw.line(screen, (0, 255, 0), (self.x, self.y), (end_x, end_y), 1)

# Oyun Döngüsü
pygame.init()
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()
arac = OtonomArac(SCREEN_WIDTH/2, SCREEN_HEIGHT/2)

running = True
while running:
    screen.fill((50, 50, 50)) # Asfalt rengi
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Manuel Kontrol (Test için)
    keys = pygame.key.get_pressed()
    if keys[pygame.K_UP]: arac.velocity += 0.1
    elif keys[pygame.K_DOWN]: arac.velocity -= 0.1
    else: arac.velocity *= 0.95 # Sürtünme
    
    if keys[pygame.K_LEFT]: arac.steering_angle = -5
    elif keys[pygame.K_RIGHT]: arac.steering_angle = 5
    else: arac.steering_angle = 0

    arac.update()
    arac.draw(screen)
    
    pygame.display.flip()
    clock.tick(60)

pygame.quit()

C:\Users\90534\AppData\Roaming\Python\Python312\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html
